# Construção de sistema de RAG utilizando Ollama

## 1) Carregamento de bibliotecas

In [45]:
import warnings

warnings.filterwarnings('ignore')

from langchain_community.document_loaders import DirectoryLoader
from langchain_text_splitters import CharacterTextSplitter
from langchain_ollama import OllamaEmbeddings
from langchain_ollama.llms import OllamaLLM
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_classic.retrievers import MultiQueryRetriever
from langchain_core.output_parsers import StrOutputParser, CommaSeparatedListOutputParser
from langchain_community.vectorstores import FAISS
from transformers import AutoTokenizer
from operator import itemgetter
from dotenv import load_dotenv
import httpx
import os

http_client = httpx.Client(verify=False)

In [2]:
os.chdir(r'c:\Users\francisco.bneto\Documents\gen-ai-formation')
print(os.getcwd())

c:\Users\francisco.bneto\Documents\gen-ai-formation


In [41]:
_ = load_dotenv()
print(f"Variáveis de ambiente carregadas: {os.getenv('OPENROUTER_API_KEY')[:10]}...")

Variáveis de ambiente carregadas: sk-or-v1-c...


## 2) Carregamento de dados

In [3]:
pdfs = DirectoryLoader("./documentos", glob="*.pdf").load()

len(pdfs)

3

## 3) Criação dos chunks

In [4]:
tokenizer = AutoTokenizer.from_pretrained("BAAI/bge-m3")

In [5]:
splitter = CharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=1250,
    chunk_overlap=150
)

chunks = splitter.split_documents(pdfs)

len(chunks)

37

## 4) Criação do banco vetorial

In [6]:
embed_model= OllamaEmbeddings(
    model="bge-m3:567m"
)

vector_store = FAISS.from_documents(
    documents=chunks,
    embedding=embed_model
)

In [7]:
prompt = """
Você é um especialista em questões bancárias, especialmente em dúvidas relacionadas a cartões de crédito.

Responsa as perguntas usando exclusivamente os conteúdos fornecidos.

Contexto:
{contexto}
"""

prompt_template = ChatPromptTemplate.from_messages(
    [
        ("system", prompt),
        ("human", "{query}")
    ]
)

llm_model = OllamaLLM(
    model="gemma3:4b"
)

In [10]:
query = "como fazer um seguro viagem?"

llm_model.invoke(query)

'Fazer um seguro viagem é uma ótima maneira de se proteger contra imprevistos durante uma viagem, seja por problemas de saúde, perda de bagagem ou cancelamento de voos. Aqui está um guia passo a passo de como fazer um seguro viagem:\n\n**1. Entenda Suas Necessidades:**\n\n*   **Destino:** O país de destino é importante porque os custos médicos e as regulamentações podem variar.\n*   **Duração da Viagem:** A duração da viagem influencia diretamente no preço do seguro.\n*   **Tipo de Viagem:** Viagens de lazer, negócios, estudos ou esportes exigem coberturas diferentes.\n*   **Número de Viajantes:** Geralmente, o preço é por pessoa.\n*   **Atividades:** Se você pretende praticar esportes de aventura, mergulho, ou outras atividades de risco, é essencial contratar um seguro que cubra essas atividades.\n*   **Coberturas Essenciais:**\n    *   **Assistência Médica e Hospitalar:** A cobertura mais importante, cobrindo despesas médicas, hospitalares, odontológicas e atendimentos de emergência.

## 5) Construção da chain

In [8]:
query = "como fazer um seguro viagem?"

retriever = vector_store.as_retriever()
trechos = retriever.invoke(query)

contexto = "\n\n".join(trecho.page_content for trecho in trechos)

chain = prompt_template | llm_model | StrOutputParser()

In [9]:
resp = chain.invoke(
    {
        "query": query,
        "contexto": contexto
    }
)

In [10]:
print(resp)

Com base no texto fornecido, aqui está um guia sobre como adquirir o seguro viagem MasterAssist Plus™:

**Requisitos:**

*   **Cartão Mastercard Platinum™:** Você precisa ser portador de um cartão Mastercard Platinum™.
*   **Dependente(s):** Você também pode incluir seus dependentes (cônjuge ou companheiro(a) e filhos dependentes) no seguro.

**Processo de Obtenção:**

1.  **Bilhete de Seguro:** Você precisa emitir um Bilhete de Seguro através do portal online: [www.aig.com/Mastercard/pt](http://www.aig.com/Mastercard/pt).
2.  **Atualização:** Mantenha seu Bilhete de Seguro atualizado sempre que houver mudanças em seus dependentes.
3.  **Coberturas:** O seguro oferece diversas coberturas, incluindo:
    *   Despesas médicas e hospitalares em viagem ao exterior (acidentes ou doenças súbitas)
    *   Traslado médico (remoção médica)
    *   Traslado de corpo (repatriação funerária)
    *   Retorno de menores/idosos
    *   Acompanhante em caso de hospitalização prolongada
    *   Hospeda

## 6) Chain com retrieval incorporado

In [11]:
rag_chain = (
    {
        "contexto": itemgetter("query") | retriever,
        "query": itemgetter("query")
    }
    | prompt_template | llm_model | StrOutputParser()
)

In [12]:
resp2 = rag_chain.invoke(
    {
        "query": query
    }
)

In [13]:
print(resp2)

De acordo com os documentos fornecidos, o seguro viagem MasterAssist Plus™ para portadores de cartão Mastercard Platinum™ oferece os seguintes benefícios e como adquirir a cobertura:

**Como Obter a Cobertura:**

1.  **Ser Elegível:** Você precisa ser portador de um cartão Mastercard Platinum™ ou membro da família, pois é o principal benefício oferecido a esses portadores.

2.  **Bilhete de Seguro:** Você precisa emitir um Bilhete de Seguro através do portal [www.aig.com/Mastercard/pt](http://www.aig.com/Mastercard/pt). Este documento formaliza sua cobertura.

3.  **Inclua seus Dependentes:** Certifique-se de que seus dependentes (cônjuges, companheiros e filhos dependentes) estejam incluídos no Bilhete de Seguro, pois eles também são cobertos. Se houver alguma mudança em seus dependentes, reemitir o Bilhete de Seguro.

**Coberturas Oferecidas:**

*   **Despesas Médicas e Hospitalares:** Em caso de acidente ou doença súbita no exterior.
*   **Traslado Médico (Remoção Médica):** Para tr

## 7) Técnicas de Retrieval

### 7.1) Rewrite-Retrieve-Read

In [53]:
query_model = OllamaLLM(
    model="gemma3:1b"
)

rewriter_prompt_template = """
Gere uma consulta de pesquisa para o banco de dados vetorial (Vector DB) a partir de uma pergunta do usuário, permitindo uma resposta mais precisa por meio da busca semântica.
Basta retornar a consulta revisada do Vector DB, entre aspas.

Pergunta do usuário: {user_question}
Consulta revisada do Vector DB:
"""

query_prompt_template = """
Você é um especialista em questões bancárias, especialmente em dúvidas relacionadas a cartões de crédito.

Responsa as perguntas usando exclusivamente os conteúdos fornecidos.

Seja breve na resposta.

Contexto:
{contexto}
"""

query = "como fazer um seguro viagem?"

In [54]:
rewriter_prompt = PromptTemplate.from_template(rewriter_prompt_template)
query_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", query_prompt_template),
        ("human", "{query}")
    ]
)

rewriter_chain = rewriter_prompt | query_model | StrOutputParser()

In [23]:
response = rewriter_chain.invoke(query)

In [24]:
print(f"Query: {query}")
print(f"Writer response: {response}")

Query: como fazer um seguro viagem?
Writer response: “seguro viagem”



In [26]:
rewriter_rag_chain = (
    {
        "contexto": itemgetter("query") | rewriter_chain | retriever,
        "query": itemgetter("query")
    } | query_prompt | llm_model | StrOutputParser()
)

In [28]:
response_rewriter = rewriter_rag_chain.invoke(
    {
        "query": query
    }
)

In [29]:
print(response_rewriter)

Com base no texto fornecido, aqui estão os passos para adquirir o seguro viagem MasterAssist Plus:

1.  **Quem está coberto:** O seguro é para portadores do cartão Mastercard Platinum, seus cônjuges, companheiros (se estiverem viajando juntos) e filhos dependentes.

2.  **Cobertura:** O seguro oferece uma série de benefícios, incluindo:
    *   **Despesas Médicas e Hospitalares:** Até USD 25.000 por pessoa em caso de acidente ou doença súbita.
    *   **Traslado Médico:** Remocao médica.
    *   **Prorrogação de Estadia:** Até USD 50.000 até USD 150 por dia (por até 5 dias)
    *   **Acompanhante:** Em caso de hospitalização prolongada.
    *   **Retorno de Menores/Idosos:** Traslado.
    *   **Traslado de Corpo (Repatriação Funerária):**
    *   **Regresso Sanitário (Repatriação Médica):**
    *   **Passagem de Ida e Volta:** Até USD 10.000, USD 25.000 ou USD 50.000.

3.  **Como fazer:**
    *   **Contato:** Para utilizar os serviços, entre em contato com os Serviços de Assistência de

### 7.2) Generating multiple queries

In [46]:
api_key = os.getenv('OPENROUTER_API_KEY')
api_url = os.getenv('OPENROUTER_BASE_URL')

# utilização de modelo do Openrouter
llm_model = ChatOpenAI(
    model="meta-llama/llama-3.1-8b-instruct",
    api_key=api_key,
    base_url=api_url,
    http_client=http_client
)

In [58]:
multi_query_prompt_template = """
Você é um assistente de modelo de linguagem de IA. Sua tarefa é gerar versões diferentes da pergunta do usuário para recuperar documentos relevantes de um banco de dados vetorial. Ao gerar múltiplas perspectivas sobre a pergunta do usuário, seu objetivo é ajudar o usuário a superar algumas das limitações da busca por similaridade baseada em distância.
Forneça estas perguntas alternativas separadas por quebras de linha. Retorne na resposta estritamente as perguntas solicitadas, sem nada adicional.
Pergunta original: {question}
"""

multi_query_prompt = PromptTemplate.from_template(multi_query_prompt_template)

multi_query_chain = multi_query_prompt | llm_model | CommaSeparatedListOutputParser()

In [59]:
print("Query: ", query)
multi_query_chain.invoke(query)

Query:  como fazer um seguro viagem?


['Como planejar e compilar um seguro para uma viagem?',
 'Quais são as etapas necessárias para adquirir um seguro de viagem?',
 'Por que um seguro viagem é importante durante uma viagem internacional?',
 'Como escolher um seguro viagem adequado para minhas necessidades?',
 'O que inclui no plano de seguro viagem para um destino específico?',
 'Existem outras formas de proteger meu investimento ao viajar além de um seguro?']

In [60]:
multi_retriever = MultiQueryRetriever(
    retriever=retriever,
    llm_chain=multi_query_chain
)

multi_rag_chain = (
    {
        "contexto": itemgetter("query") | multi_retriever,
        "query": itemgetter("query")
    } | query_prompt | llm_model | StrOutputParser()
)

In [61]:
response_multi_query = multi_rag_chain.invoke(
    {
        "query": query
    }
)

In [62]:
print(response_multi_query)

Para fazer um seguro viagem, você precisa seguir os seguintes passos:

1. **Verificar a elegibilidade**: Verifique se você está elegível para o seguro viagem. Geralmente, é necessário ter um cartão de crédito Mastercard Platinum ou outro cartão qualificado.
2. **Informar a viagem**: Informe a Mastercard sobre a viagem que você está planejando. Isso pode ser feito através do site MyCardBenefits.com ou pelo telefone.
3. **Obter o Bilhete de Seguro**: Você precisará emitir o Bilhete de Seguro Anual, que é um documento obrigatório para obter a cobertura do seguro viagem.
4. **Verificar as condições**: Leia atentamente as condições do seguro viagem, incluindo as exclusões e os limites de cobertura.
5. **Pagar a taxa de seguro**: Você precisará pagar a taxa de seguro, que é geralmente uma taxa única ou uma taxa mensal.

É importante notar que o seguro viagem não é um seguro saúde, e as coberturas são limitadas. É importante ler atentamente as condições e os termos antes de emitir o Bilhete d

### 7.3) HyDE: Hypothetical Document Embeddings

In [63]:
hyde_prompt_template = """
Escreva um parágrafo que possa responder a pergunta apresentada. Não adicione informações que não estão relacionadas a pergunta.
Pergunta: {user_question}
Paragrafo:
"""

hyde_prompt = PromptTemplate.from_template(
    hyde_prompt_template
)

hyde_chain = hyde_prompt | llm_model | StrOutputParser()

hyde_chain.invoke(query)

'Para fazer um seguro de viagem é necessário escolher o tipo de seguro que melhor se adapte às suas necessidades, seja o seguro de saúde, o seguro de bagagem ou o seguro de cancelamento. Após escolher o tipo de seguro, é necessário preencher o formulário com todas as informações necessárias, como dados pessoais e detalhes da viagem. Além disso, é importante verificar se o seguro é válido para o seu destino de viagem e se há algum tipo de exclusão. Após preencher o formulário, é necessário pagar a taxa do seguro, que pode ser feita através de cartão de crédito ou transferência bancária. Após pagar a taxa, é importante ler o contrato do seguro e entender todas as cláusulas e condições para que possa ter uma proteção adequada durante a sua viagem.'

In [64]:
hyde_rag_chain = (
    {
        "contexto": itemgetter("query") | hyde_chain | retriever,
        "query": itemgetter("query")
    } | query_prompt | llm_model | StrOutputParser()
)

In [65]:
response_hyde = hyde_rag_chain.invoke(
    {
        "query": query
    }
)

In [66]:
print(response_hyde)

Para fazer um seguro viagem com o Mastercard Platinum, é necessário seguir os passos abaixo:

1. Verifique se você tem direito a cobertura: Verifique se você é portador de um cartão Mastercard Platinum válido e se a viagem que você está planejando está coberta pelo seguro.
2. Emissão do Bilhete de Seguro: Emite o Bilhete de Seguro através do portal www.aig.com/Mastercard/pt. Esse documento é necessário para comprovar a cobertura em caso de ocorrência/sinistro.
3. Preencha o Formulário de Sinistro: Em caso de ocorrência/sinistro, preencha o Formulário de Sinistro e envie todas as informações exigidas, incluindo cópias de documentos que comprovem a perda.
4. Envie as informações necessárias: Envie as informações necessárias para o processamento do sinistro, incluindo cópias de documentos que comprovem a perda.
5. Avisar a Mastercard: Avise a Mastercard sobre a ocorrência/sinistro e forneça todas as informações solicitadas.

Observações:

* A cobertura é válida apenas para viagens interna